# Benchmark hybrid fix notebook cells

This notebook patch keeps the same overall style and function structure as the earlier file, but makes the hybrid section more robust.

Main fixes:
- rebuilds `item_meta` safely even after a partial kernel restart
- rebuilds feature tensors from saved benchmark files when needed
- loads the base checkpoint safely across PyTorch versions
- removes the duplicated stage-2 training block
- saves clean outputs for the hybrid run

## A. Small inspection cells first

Run these before the fixed hybrid cells. They help find the real cause fast without changing the main pipeline.

In [1]:
# A1 — inspect current environment and globals
import torch
import pandas as pd

print("torch version:", torch.__version__)
print("has movies in globals:", "movies" in globals())
print("has tags in globals:", "tags" in globals())
print("has item_map in globals:", "item_map" in globals())
print("has interactions in globals:", "interactions" in globals())
print("has splits in globals:", "splits" in globals())
print("has num_items in globals:", "num_items" in globals())
print("has MODEL_DIR in globals:", "MODEL_DIR" in globals())
print("has PROC_DIR in globals:", "PROC_DIR" in globals())

torch version: 2.5.1+cu124
has movies in globals: False
has tags in globals: False
has item_map in globals: False
has interactions in globals: False
has splits in globals: False
has num_items in globals: False
has MODEL_DIR in globals: False
has PROC_DIR in globals: False


In [2]:
# A2 — inspect saved benchmark files
from pathlib import Path

PROC_DIR = Path("../data/processed_benchmark")
MODEL_DIR = Path("../models")
RESULT_DIR = Path("../reports/results")

print("PROC_DIR exists:", PROC_DIR.exists())
print("MODEL_DIR exists:", MODEL_DIR.exists())
print("RESULT_DIR exists:", RESULT_DIR.exists())

if PROC_DIR.exists():
    print("processed files:", sorted([p.name for p in PROC_DIR.iterdir()]))

if MODEL_DIR.exists():
    print("model files:", sorted([p.name for p in MODEL_DIR.iterdir()])[:20])

PROC_DIR exists: True
MODEL_DIR exists: True
RESULT_DIR exists: True
processed files: ['interactions_full_positive_u5.pkl', 'item_map_full_positive_u5.pkl', 'splits_full_positive_u5.pkl', 'user_map_full_positive_u5.pkl']
model files: ['benchmark_sasrec_base.pt', 'sasrec_base_full_positive_u5.pt', 'sasrec_base_id_only.pt', 'sasrec_structured.pt', 'sasrec_structured_gated.pt', 'sasrec_structured_text_gated.pt']


In [3]:
# A3 — inspect the base checkpoint keys
from pathlib import Path

base_ckpt_path = Path("../models/benchmark_sasrec_base.pt")
print("base checkpoint exists:", base_ckpt_path.exists())

if base_ckpt_path.exists():
    try:
        base_state_preview = torch.load(base_ckpt_path, map_location="cpu", weights_only=True)
    except TypeError:
        base_state_preview = torch.load(base_ckpt_path, map_location="cpu")

    print("num keys:", len(base_state_preview))
    print("first 20 keys:")
    for k in list(base_state_preview.keys())[:20]:
        print(" ", k)

base checkpoint exists: True
num keys: 54
first 20 keys:
  item_emb.weight
  pos_emb.weight
  emb_norm.weight
  emb_norm.bias
  encoder.layers.0.self_attn.in_proj_weight
  encoder.layers.0.self_attn.in_proj_bias
  encoder.layers.0.self_attn.out_proj.weight
  encoder.layers.0.self_attn.out_proj.bias
  encoder.layers.0.linear1.weight
  encoder.layers.0.linear1.bias
  encoder.layers.0.linear2.weight
  encoder.layers.0.linear2.bias
  encoder.layers.0.norm1.weight
  encoder.layers.0.norm1.bias
  encoder.layers.0.norm2.weight
  encoder.layers.0.norm2.bias
  encoder.layers.1.self_attn.in_proj_weight
  encoder.layers.1.self_attn.in_proj_bias
  encoder.layers.1.self_attn.out_proj.weight
  encoder.layers.1.self_attn.out_proj.bias


## B. Fixed hybrid benchmark cells

These cells replace the old `S` to final hybrid cells.

In [4]:

print("user_map in globals:", "user_map" in globals())
if "user_map" in globals():
    print("type(user_map):", type(user_map))
    print("user_map is None:", user_map is None)

print("item_map in globals:", "item_map" in globals())
if "item_map" in globals():
    print("type(item_map):", type(item_map))
    print("item_map is None:", item_map is None)

user_map in globals: False
item_map in globals: False


In [5]:
# S — rebuild benchmark objects safely when the kernel was partially restarted
import math
import time
import random
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

DATA_DIR = Path("../data/ml-20m")
PROC_DIR = Path("../data/processed_benchmark")
MODEL_DIR = Path("../models")
RESULT_DIR = Path("../reports/results")
TEXT_DIR = Path("../data/processed")

PROC_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)
TEXT_DIR.mkdir(parents=True, exist_ok=True)

MAX_LEN = 200
D_MODEL = 256
N_HEADS = 4
N_LAYERS = 4
DROPOUT = 0.1
BATCH_SIZE = 128
LR = 1e-3
WEIGHT_DECAY = 1e-5
GRAD_CLIP = 1.0

def need_reload(name):
    return (name not in globals()) or (globals()[name] is None)

if need_reload("interactions"):
    p = PROC_DIR / "interactions_full_positive_u5.pkl"
    if p.exists():
        interactions = pd.read_pickle(p)
    else:
        raise FileNotFoundError(f"Missing file: {p}")

if need_reload("splits"):
    p = PROC_DIR / "splits_full_positive_u5.pkl"
    if p.exists():
        splits = pd.read_pickle(p)
    else:
        raise FileNotFoundError(f"Missing file: {p}")

if need_reload("item_map"):
    p = PROC_DIR / "item_map_full_positive_u5.pkl"
    if p.exists():
        item_map = pd.read_pickle(p)
    else:
        raise FileNotFoundError(f"Missing file: {p}")

if need_reload("user_map"):
    p = PROC_DIR / "user_map_full_positive_u5.pkl"
    if p.exists():
        user_map = pd.read_pickle(p)
    else:
        user_map = None

if "movies" not in globals() or movies is None:
    movies = pd.read_csv(DATA_DIR / "movies.csv")

if "tags" not in globals() or tags is None:
    tags = pd.read_csv(DATA_DIR / "tags.csv")

if ("user_map" in globals()) and (user_map is not None):
    num_users = len(user_map)
else:
    if "splits" in globals() and splits is not None and "user_id" in splits.columns:
        num_users = int(splits["user_id"].max())
    elif "interactions" in globals() and interactions is not None and "user_id" in interactions.columns:
        num_users = int(interactions["user_id"].max())
    else:
        raise ValueError("Could not recover num_users because user_map and fallback user_id columns are unavailable.")

if ("item_map" in globals()) and (item_map is not None):
    num_items = len(item_map)
else:
    if "splits" in globals() and splits is not None and "item_id" in splits.columns:
        num_items = int(splits["item_id"].max())
    elif "interactions" in globals() and interactions is not None and "item_id" in interactions.columns:
        num_items = int(interactions["item_id"].max())
    else:
        raise ValueError("Could not recover num_items because item_map and fallback item_id columns are unavailable.")

print("num_users:", num_users)
print("num_items:", num_items)
print("splits shape:", splits.shape)
print("movies shape:", movies.shape)
print("tags shape:", tags.shape)

device: cuda
num_users: 138493
num_items: 26744
splits shape: (138493, 7)
movies shape: (27278, 3)
tags shape: (465564, 4)


In [6]:
# S0 — rebuild item_meta robustly
if isinstance(item_map, pd.DataFrame):
    item_map_df = item_map.copy()
else:
    item_map_df = pd.DataFrame(item_map)

if "movieId" not in item_map_df.columns or "item_idx" not in item_map_df.columns:
    if {"movieId", "item_idx"}.issubset(interactions.columns):
        item_map_df = (
            interactions[["movieId", "item_idx"]]
            .drop_duplicates()
            .sort_values("item_idx")
            .reset_index(drop=True)
        )
    else:
        raise ValueError("Need item_map with columns movieId and item_idx")

movies = movies.copy()
movies["year"] = pd.to_numeric(
    movies["title"].astype(str).str.extract(r"\((\d{4})\)")[0],
    errors="coerce"
)
movies["clean_title"] = movies["title"].astype(str).str.replace(r"\s*\(\d{4}\)$", "", regex=True)
movies["genres_text"] = movies["genres"].fillna("").str.replace("|", " ", regex=False)

tags = tags.copy()
tags = tags.dropna(subset=["tag"]).copy()
tags["tag_lower"] = tags["tag"].astype(str).str.strip().str.lower()
tags = tags[tags["tag_lower"] != ""].copy()

tag_counts = (
    tags.groupby(["movieId", "tag_lower"])
        .size()
        .reset_index(name="count")
        .sort_values(["movieId", "count", "tag_lower"], ascending=[True, False, True])
)

top_tags = tag_counts.groupby("movieId").head(5).copy()

top_tags_text = (
    top_tags.groupby("movieId")["tag_lower"]
    .apply(lambda s: " | ".join(s.tolist()))
    .reset_index(name="top_tags_text")
)

item_meta = (
    item_map_df.merge(
        movies[["movieId", "title", "clean_title", "genres", "genres_text", "year"]],
        on="movieId",
        how="left"
    )
    .merge(
        top_tags_text,
        on="movieId",
        how="left"
    )
)

item_meta["top_tags_text"] = item_meta["top_tags_text"].fillna("")
item_meta["movie_text"] = (
    item_meta["clean_title"].fillna("") + ". " +
    item_meta["genres_text"].fillna("") + ". " +
    item_meta["top_tags_text"].fillna("")
).str.strip()

item_meta = item_meta.sort_values("item_idx").reset_index(drop=True)

print("item_meta shape:", item_meta.shape)
print(
    item_meta[["item_idx", "movieId", "clean_title", "genres", "top_tags_text"]]
    .head(10)
    .to_string(index=False)
)

item_meta shape: (26744, 9)
 item_idx  movieId                 clean_title                                      genres                                                                         top_tags_text
        1        1                   Toy Story Adventure|Animation|Children|Comedy|Fantasy                           pixar | animation | disney | tom hanks | computer animation
        2        2                     Jumanji                  Adventure|Children|Fantasy                         robin williams | fantasy | time travel | animals | board game
        3        3            Grumpier Old Men                              Comedy|Romance                        moldy | old | sequel | clv | comedinha de velhinhos engraã§ada
        4        4           Waiting to Exhale                        Comedy|Drama|Romance                                              characters | chick flick | clv | revenge
        5        5 Father of the Bride Part II                                      Com

In [7]:
# S1 — build structured feature tensors
genre_dummies = item_meta["genres"].fillna("").str.get_dummies(sep="|")
genre_cols = genre_dummies.columns.tolist()

year_vocab = [
    "unknown",
    "<=1950",
    "1951-1960",
    "1961-1970",
    "1971-1980",
    "1981-1990",
    "1991-2000",
    "2001-2010",
    "2011-2020",
]
year2idx = {y: i for i, y in enumerate(year_vocab)}

item_meta["year_bucket"] = pd.cut(
    item_meta["year"],
    bins=[0, 1950, 1960, 1970, 1980, 1990, 2000, 2010, 2020],
    labels=year_vocab[1:],
    include_lowest=True
).astype("object").fillna("unknown")

genre_mat = np.zeros((num_items + 1, len(genre_cols)), dtype=np.float32)
year_ids = np.zeros(num_items + 1, dtype=np.int64)

idx = item_meta["item_idx"].to_numpy()
genre_mat[idx] = genre_dummies.to_numpy(dtype=np.float32)
year_ids[idx] = item_meta["year_bucket"].map(year2idx).fillna(0).astype(int).to_numpy()

genre_tensor = torch.tensor(genre_mat, dtype=torch.float32)
year_tensor = torch.tensor(year_ids, dtype=torch.long)

print("genre tensor:", genre_tensor.shape)
print("year tensor:", year_tensor.shape)
print("genre cols:", len(genre_cols))

genre tensor: torch.Size([26745, 20])
year tensor: torch.Size([26745])
genre cols: 20


In [8]:
# T — load or build text embeddings
import re
import unicodedata
from sentence_transformers import SentenceTransformer

TEXT_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
TEXT_BATCH_SIZE = 256
TEXT_PATH = TEXT_DIR / "benchmark_text_minilm_tensor.pt"

def clean_text_basic(x):
    x = "" if pd.isna(x) else str(x)
    x = unicodedata.normalize("NFKC", x)
    x = x.replace("|", " ")
    x = re.sub(r"\s+", " ", x).strip().lower()
    return x

if TEXT_PATH.exists():
    text_tensor = torch.load(TEXT_PATH, map_location="cpu")
    print("loaded existing text tensor:", TEXT_PATH)
    print("text tensor shape:", text_tensor.shape)
else:
    item_text_df = item_meta[["item_idx", "movieId", "clean_title", "genres_text", "top_tags_text"]].copy()

    item_text_df["movie_text_clean"] = (
        item_text_df["clean_title"].fillna("").map(clean_text_basic) + ". " +
        item_text_df["genres_text"].fillna("").map(clean_text_basic) + ". " +
        item_text_df["top_tags_text"].fillna("").map(clean_text_basic)
    ).str.strip()

    text_model = SentenceTransformer(TEXT_MODEL_NAME)

    texts = item_text_df.sort_values("item_idx")["movie_text_clean"].tolist()

    text_emb = text_model.encode(
        texts,
        batch_size=TEXT_BATCH_SIZE,
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=True
    )

    text_emb_full = np.zeros((num_items + 1, text_emb.shape[1]), dtype=np.float32)
    text_emb_full[1:] = text_emb.astype(np.float32)

    text_tensor = torch.tensor(text_emb_full, dtype=torch.float32)
    torch.save(text_tensor, TEXT_PATH)

    print("built and saved text tensor:", TEXT_PATH)
    print("text tensor shape:", text_tensor.shape)

/workspace/movie-transformer-recommender/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


loaded existing text tensor: ../data/processed/benchmark_text_minilm_tensor.pt
text tensor shape: torch.Size([26745, 384])


/tmp/ipykernel_3783/500305022.py:18: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  text_tensor = torch.load(TEXT_PATH, map_location="cpu")


In [9]:
# U — rebuild datasets and loaders
def pad_seq(seq, max_len=200):
    seq = seq[-max_len:]
    out = np.zeros(max_len, dtype=np.int64)
    out[-len(seq):] = seq
    return out

class TrainDataset(Dataset):
    def __init__(self, splits_df, max_len=200):
        self.max_len = max_len
        self.seqs = splits_df["train_seq"].tolist()

        keep = []
        for seq in self.seqs:
            if len(seq) >= 3:
                keep.append(seq)

        self.data = keep

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        seq = self.data[idx]

        tokens = seq[:-1]
        targets = seq[1:]

        tokens = pad_seq(tokens, self.max_len)
        targets = pad_seq(targets, self.max_len)

        return {
            "seq": torch.tensor(tokens, dtype=torch.long),
            "target_seq": torch.tensor(targets, dtype=torch.long),
        }

class EvalDataset(Dataset):
    def __init__(self, splits_df, mode="val", max_len=200):
        self.max_len = max_len

        if mode == "val":
            self.seq_col = "val_seq"
            self.target_col = "val_target"
        else:
            self.seq_col = "test_seq"
            self.target_col = "test_target"

        self.seq = splits_df[self.seq_col].tolist()
        self.target = splits_df[self.target_col].tolist()

    def __len__(self):
        return len(self.seq)

    def __getitem__(self, idx):
        return {
            "seq": torch.tensor(pad_seq(self.seq[idx], self.max_len), dtype=torch.long),
            "target": torch.tensor(self.target[idx], dtype=torch.long),
        }

train_ds = TrainDataset(splits, max_len=MAX_LEN)
val_ds = EvalDataset(splits, mode="val", max_len=MAX_LEN)
test_ds = EvalDataset(splits, mode="test", max_len=MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print("train sequences:", len(train_ds))
print("val users:", len(val_ds))
print("test users:", len(test_ds))

train sequences: 138493
val users: 138493
test users: 138493


In [ ]:
# V — model and helper functions
import matplotlib.pyplot as plt

USE_AMP = torch.cuda.is_available()
LABEL_SMOOTHING = 0.05
ACCUM_STEPS = 2
MIN_DELTA = 1e-4


def build_optimizer(model, lr=1e-3, weight_decay=1e-4):
    return torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)


def build_hybrid_optimizer(model, backbone_lr, head_lr, weight_decay=1e-4):
    head_names = [
        "genre_proj",
        "year_emb",
        "text_proj",
        "struct_norm",
        "text_norm",
        "struct_gate",
        "text_gate",
    ]

    backbone_params = []
    head_params = []

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if any(name.startswith(prefix) for prefix in head_names):
            head_params.append(param)
        else:
            backbone_params.append(param)

    param_groups = []
    if backbone_params:
        param_groups.append({"params": backbone_params, "lr": backbone_lr})
    if head_params:
        param_groups.append({"params": head_params, "lr": head_lr})

    return torch.optim.AdamW(param_groups, weight_decay=weight_decay)


def build_scheduler(optimizer, steps_per_epoch, epochs, max_lr):
    if isinstance(max_lr, (list, tuple)):
        max_lr_list = list(max_lr)
    else:
        max_lr_list = [max_lr for _ in optimizer.param_groups]

    return torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=max_lr_list,
        epochs=epochs,
        steps_per_epoch=steps_per_epoch,
        pct_start=0.10,
        anneal_strategy="cos",
        div_factor=25.0,
        final_div_factor=1e3,
    )


def train_one_epoch(
    model,
    loader,
    optimizer,
    device,
    scheduler=None,
    scaler=None,
    grad_clip=1.0,
    accumulation_steps=1,
    label_smoothing=0.0,
):
    model.train()
    total_loss = 0.0
    total_tokens = 0

    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(loader, start=1):
        seq = batch["seq"].to(device, non_blocking=True)
        target_seq = batch["target_seq"].to(device, non_blocking=True)

        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(scaler is not None)):
            logits = model.predict_all_positions(seq)
            ce_loss = nn.functional.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                target_seq.reshape(-1),
                ignore_index=0,
                label_smoothing=label_smoothing,
            )
            loss = ce_loss / accumulation_steps

        if scaler is not None:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        if step % accumulation_steps == 0 or step == len(loader):
            if scaler is not None:
                scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

            if scaler is not None:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()

            optimizer.zero_grad(set_to_none=True)

            if scheduler is not None:
                scheduler.step()

        valid_tokens = (target_seq > 0).sum().item()
        total_loss += ce_loss.detach().float().item() * valid_tokens
        total_tokens += valid_tokens

    return total_loss / max(total_tokens, 1)


@torch.no_grad()
def evaluate_topk(model, loader, device, k=10):
    model.eval()

    hits = 0.0
    ndcg = 0.0
    mrr = 0.0
    total = 0

    for batch in loader:
        seq = batch["seq"].to(device, non_blocking=True)
        target = batch["target"].to(device, non_blocking=True)

        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model.predict_all(seq)

        for i in range(seq.size(0)):
            seen = seq[i].unique()
            seen = seen[seen > 0]
            logits[i, seen] = -1e9

        topk = torch.topk(logits, k=k, dim=1).indices

        for i in range(seq.size(0)):
            tgt = target[i].item()
            pred = topk[i].tolist()

            if tgt in pred:
                rank = pred.index(tgt) + 1
                hits += 1.0
                ndcg += 1.0 / math.log2(rank + 1)
                mrr += 1.0 / rank

        total += seq.size(0)

    hr = hits / total
    ndcg = ndcg / total
    mrr = mrr / total

    return {
        f"HR@{k}": hr,
        f"NDCG@{k}": ndcg,
        f"MRR@{k}": mrr,
        f"Recall@{k}": hr,
    }


def plot_history_curves(history_df, title_prefix, x_col="epoch"):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(history_df[x_col], history_df["train_loss"], label="train_loss")
    ax.set_title(f"{title_prefix} train loss")
    ax.set_xlabel(x_col)
    ax.set_ylabel("loss")
    ax.legend()
    plt.show()

    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(history_df[x_col], history_df["NDCG@10"], label="NDCG@10")
    ax.plot(history_df[x_col], history_df["HR@10"], label="HR@10")
    ax.plot(history_df[x_col], history_df["MRR@10"], label="MRR@10")
    ax.set_title(f"{title_prefix} validation metrics")
    ax.set_xlabel(x_col)
    ax.set_ylabel("metric")
    ax.legend()
    plt.show()


def load_submodule_from_flat_state(submodule, state_dict, prefix):
    sub_state = {}
    for k, v in state_dict.items():
        if k.startswith(prefix):
            sub_state[k[len(prefix):]] = v
    submodule.load_state_dict(sub_state)

In [11]:
# W — build hybrid model and warm-start from the best base checkpoint
hybrid_model = SASRecHybridGated(
    num_items=num_items,
    genre_tensor=genre_tensor,
    year_tensor=year_tensor,
    text_tensor=text_tensor,
    max_len=MAX_LEN,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    dropout=DROPOUT,
).to(device)

base_ckpt_path = MODEL_DIR / "benchmark_sasrec_base.pt"

try:
    base_state = torch.load(base_ckpt_path, map_location="cpu", weights_only=True)
except TypeError:
    base_state = torch.load(base_ckpt_path, map_location="cpu")

hybrid_model.item_emb.load_state_dict({
    "weight": base_state["item_emb.weight"]
})
hybrid_model.pos_emb.load_state_dict({
    "weight": base_state["pos_emb.weight"]
})
hybrid_model.emb_norm.load_state_dict({
    "weight": base_state["emb_norm.weight"],
    "bias": base_state["emb_norm.bias"],
})
hybrid_model.final_norm.load_state_dict({
    "weight": base_state["final_norm.weight"],
    "bias": base_state["final_norm.bias"],
})

for i in range(N_LAYERS):
    load_submodule_from_flat_state(
        hybrid_model.encoder.layers[i],
        base_state,
        prefix=f"encoder.layers.{i}."
    )

print(hybrid_model)

/workspace/movie-transformer-recommender/.venv/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


SASRecHybridGated(
  (item_emb): Embedding(26745, 256, padding_idx=0)
  (pos_emb): Embedding(201, 256, padding_idx=0)
  (genre_proj): Linear(in_features=20, out_features=256, bias=True)
  (year_emb): Embedding(9, 256)
  (text_proj): Linear(in_features=384, out_features=256, bias=True)
  (struct_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (text_norm): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
  (struct_dropout): Dropout(p=0.1, inplace=False)
  (text_dropout): Dropout(p=0.1, inplace=False)
  (struct_gate): Sequential(
    (0): Linear(in_features=512, out_features=256, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): Sigmoid()
  )
  (text_gate): Sequential(
    (0): Linear(in_features=512, out_features=256, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): Sigmoid()
  )
  (emb_dropout): Dropout(p=0.1, inplace=False)
  (emb_norm

In [12]:

# inspect

print("before cleanup")
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("allocated GB:", torch.cuda.memory_allocated() / 1024**3)
    print("reserved  GB:", torch.cuda.memory_reserved() / 1024**3)

torch.cuda.empty_cache()

print("\nafter empty_cache")
if torch.cuda.is_available():
    print("allocated GB:", torch.cuda.memory_allocated() / 1024**3)
    print("reserved  GB:", torch.cuda.memory_reserved() / 1024**3)



before cleanup
cuda available: True
allocated GB: 0.08059263229370117
reserved  GB: 0.1015625

after empty_cache
allocated GB: 0.08059263229370117
reserved  GB: 0.1015625


In [1]:
# X — stage 1 fine-tune new hybrid heads only

# REPLACE CELL X — stage 1 fine-tune new hybrid heads only
for p in hybrid_model.item_emb.parameters():
    p.requires_grad = False
for p in hybrid_model.pos_emb.parameters():
    p.requires_grad = False
for p in hybrid_model.encoder.parameters():
    p.requires_grad = False
for p in hybrid_model.emb_norm.parameters():
    p.requires_grad = False
for p in hybrid_model.final_norm.parameters():
    p.requires_grad = False

LR_STAGE1 = 5e-4
WEIGHT_DECAY = 1e-4
EPOCHS_STAGE1 = 8
PATIENCE_STAGE1 = 8
MIN_DELTA_STAGE1 = 1e-4
ACCUM_STEPS = 2
LABEL_SMOOTHING = 0.05
GRAD_CLIP = 1.0

optimizer = build_optimizer(hybrid_model, lr=LR_STAGE1, weight_decay=WEIGHT_DECAY)
steps_per_epoch_stage1 = math.ceil(len(train_loader) / ACCUM_STEPS)
scheduler = build_scheduler(
    optimizer,
    steps_per_epoch=steps_per_epoch_stage1,
    epochs=EPOCHS_STAGE1,
    max_lr=LR_STAGE1,
)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

history_hybrid_stage1 = []
best_val_ndcg_hybrid = -1.0
best_state_hybrid = None
bad_epochs_stage1 = 0

train_start_stage1 = time.perf_counter()

for epoch in range(1, EPOCHS_STAGE1 + 1):
    t0 = time.perf_counter()

    train_loss = train_one_epoch(
        hybrid_model,
        train_loader,
        optimizer,
        device,
        scheduler=scheduler,
        scaler=scaler,
        grad_clip=GRAD_CLIP,
        accumulation_steps=ACCUM_STEPS,
        label_smoothing=LABEL_SMOOTHING,
    )
    val_metrics = evaluate_topk(hybrid_model, val_loader, device, k=10)

    epoch_time = time.perf_counter() - t0
    current_val_ndcg = val_metrics["NDCG@10"]
    lr_now = optimizer.param_groups[0]["lr"]

    row = {
        "stage": 1,
        "epoch": epoch,
        "train_loss": train_loss,
        "epoch_sec": epoch_time,
        "lr": lr_now,
        **val_metrics,
    }
    history_hybrid_stage1.append(row)

    print(
        f"stage1 epoch {epoch:03d} | "
        f"loss {train_loss:.4f} | "
        f"HR@10 {val_metrics['HR@10']:.4f} | "
        f"NDCG@10 {val_metrics['NDCG@10']:.4f} | "
        f"MRR@10 {val_metrics['MRR@10']:.4f} | "
        f"lr {lr_now:.6f} | "
        f"bad {bad_epochs_stage1}/{PATIENCE_STAGE1} | "
        f"{epoch_time:.1f}s"
    )

    if current_val_ndcg > best_val_ndcg_hybrid + MIN_DELTA_STAGE1:
        best_val_ndcg_hybrid = current_val_ndcg
        best_state_hybrid = {k: v.detach().cpu().clone() for k, v in hybrid_model.state_dict().items()}
        bad_epochs_stage1 = 0
        print(f"  new best stage1 NDCG@10: {best_val_ndcg_hybrid:.6f}")
    else:
        bad_epochs_stage1 += 1

    if bad_epochs_stage1 >= PATIENCE_STAGE1:
        print(f"stage1 early stopping at epoch {epoch}")
        break

total_train_sec_stage1 = time.perf_counter() - train_start_stage1
print("\nhybrid stage1 finished in", round(total_train_sec_stage1, 2), "seconds")

if best_state_hybrid is not None:
    hybrid_model.load_state_dict(best_state_hybrid)

NameError: name 'hybrid_model' is not defined

In [ ]:

# Y — stage 2 unfreeze all and fine-tune end-to-end with long early stopping

for p in hybrid_model.parameters():
    p.requires_grad = True

if best_state_hybrid is not None:
    hybrid_model.load_state_dict(best_state_hybrid)

BACKBONE_LR_STAGE2 = 5e-5
HEAD_LR_STAGE2 = 2e-4
WEIGHT_DECAY = 1e-4
EPOCHS_STAGE2 = 240
PATIENCE = 50
MIN_DELTA = 1e-4
ACCUM_STEPS = 2
LABEL_SMOOTHING = 0.05
GRAD_CLIP = 1.0

optimizer = build_hybrid_optimizer(
    hybrid_model,
    backbone_lr=BACKBONE_LR_STAGE2,
    head_lr=HEAD_LR_STAGE2,
    weight_decay=WEIGHT_DECAY,
)
max_lr_stage2 = [group["lr"] for group in optimizer.param_groups]
steps_per_epoch_stage2 = math.ceil(len(train_loader) / ACCUM_STEPS)
scheduler = build_scheduler(
    optimizer,
    steps_per_epoch=steps_per_epoch_stage2,
    epochs=EPOCHS_STAGE2,
    max_lr=max_lr_stage2,
)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

history_hybrid_stage2 = []
best_val_ndcg_stage2 = best_val_ndcg_hybrid
best_state_stage2 = {k: v.detach().cpu().clone() for k, v in hybrid_model.state_dict().items()}
epochs_without_improvement = 0

train_start_hybrid = time.perf_counter()

for epoch in range(1, EPOCHS_STAGE2 + 1):
    t0 = time.perf_counter()

    train_loss = train_one_epoch(
        hybrid_model,
        train_loader,
        optimizer,
        device,
        scheduler=scheduler,
        scaler=scaler,
        grad_clip=GRAD_CLIP,
        accumulation_steps=ACCUM_STEPS,
        label_smoothing=LABEL_SMOOTHING,
    )
    val_metrics = evaluate_topk(hybrid_model, val_loader, device, k=10)

    epoch_time = time.perf_counter() - t0
    current_val_ndcg = val_metrics["NDCG@10"]
    lrs_now = [group["lr"] for group in optimizer.param_groups]

    row = {
        "stage": 2,
        "epoch": epoch,
        "train_loss": train_loss,
        "epoch_sec": epoch_time,
        "lr_backbone": lrs_now[0],
        "lr_heads": lrs_now[-1],
        **val_metrics,
    }
    history_hybrid_stage2.append(row)

    print(
        f"stage2 epoch {epoch:03d} | "
        f"loss {train_loss:.4f} | "
        f"HR@10 {val_metrics['HR@10']:.4f} | "
        f"NDCG@10 {val_metrics['NDCG@10']:.4f} | "
        f"MRR@10 {val_metrics['MRR@10']:.4f} | "
        f"lr_backbone {lrs_now[0]:.6f} | "
        f"lr_heads {lrs_now[-1]:.6f} | "
        f"bad {epochs_without_improvement}/{PATIENCE} | "
        f"{epoch_time:.1f}s"
    )

    if current_val_ndcg > best_val_ndcg_stage2 + MIN_DELTA:
        best_val_ndcg_stage2 = current_val_ndcg
        best_state_stage2 = {k: v.detach().cpu().clone() for k, v in hybrid_model.state_dict().items()}
        epochs_without_improvement = 0
        print(f"  new best stage2 NDCG@10: {best_val_ndcg_stage2:.6f}")
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping triggered after {PATIENCE} consecutive non-improving epochs.")
        break

total_train_sec_hybrid = time.perf_counter() - train_start_hybrid

if best_state_stage2 is not None:
    hybrid_model.load_state_dict(best_state_stage2)

best_state_hybrid = {k: v.detach().cpu().clone() for k, v in hybrid_model.state_dict().items()}
best_val_ndcg_hybrid = best_val_ndcg_stage2

print("\nhybrid fine-tuning finished in", round(total_train_sec_hybrid, 2), "seconds")



stage2 epoch 001 | loss 5.0074 | HR@10 0.0507 | NDCG@10 0.0275 | MRR@10 0.0205 | no_improve 0/5 | 157.8s
  new best stage2 NDCG@10: 0.0275
stage2 epoch 002 | loss 5.0030 | HR@10 0.0507 | NDCG@10 0.0276 | MRR@10 0.0206 | no_improve 0/5 | 157.9s
  new best stage2 NDCG@10: 0.0276
stage2 epoch 003 | loss 4.9987 | HR@10 0.0507 | NDCG@10 0.0275 | MRR@10 0.0205 | no_improve 0/5 | 158.1s
stage2 epoch 004 | loss 4.9948 | HR@10 0.0507 | NDCG@10 0.0275 | MRR@10 0.0205 | no_improve 1/5 | 158.1s
stage2 epoch 005 | loss 4.9911 | HR@10 0.0505 | NDCG@10 0.0274 | MRR@10 0.0205 | no_improve 2/5 | 158.0s
stage2 epoch 006 | loss 4.9875 | HR@10 0.0507 | NDCG@10 0.0276 | MRR@10 0.0206 | no_improve 3/5 | 157.9s
  new best stage2 NDCG@10: 0.0276
stage2 epoch 007 | loss 4.9841 | HR@10 0.0506 | NDCG@10 0.0276 | MRR@10 0.0206 | no_improve 0/5 | 158.0s
stage2 epoch 008 | loss 4.9812 | HR@10 0.0506 | NDCG@10 0.0276 | MRR@10 0.0207 | no_improve 1/5 | 158.0s
  new best stage2 NDCG@10: 0.0276
stage2 epoch 009 | loss 

In [15]:
# # Y — stage 2 unfreeze all and continue fine-tuning
# for p in hybrid_model.parameters():
#     p.requires_grad = True

# hybrid_model.load_state_dict(best_state_hybrid)

# LR_STAGE2 = 3e-4
# EPOCHS_STAGE2 = 6

# optimizer = build_optimizer(hybrid_model, lr=LR_STAGE2, weight_decay=WEIGHT_DECAY)
# scheduler = build_scheduler(optimizer, epochs=EPOCHS_STAGE2)

# history_hybrid_stage2 = []
# train_start_hybrid = time.perf_counter()

# for epoch in range(1, EPOCHS_STAGE2 + 1):
#     t0 = time.perf_counter()

#     train_loss = train_one_epoch(hybrid_model, train_loader, optimizer, device, grad_clip=GRAD_CLIP)
#     val_metrics = evaluate_topk(hybrid_model, val_loader, device, k=10)

#     scheduler.step()

#     epoch_time = time.perf_counter() - t0
#     lr_now = optimizer.param_groups[0]["lr"]

#     row = {
#         "stage": 2,
#         "epoch": epoch,
#         "train_loss": train_loss,
#         "epoch_sec": epoch_time,
#         "lr": lr_now,
#         **val_metrics
#     }
#     history_hybrid_stage2.append(row)

#     print(
#         f"stage2 epoch {epoch:02d} | "
#         f"loss {train_loss:.4f} | "
#         f"HR@10 {val_metrics['HR@10']:.4f} | "
#         f"NDCG@10 {val_metrics['NDCG@10']:.4f} | "
#         f"MRR@10 {val_metrics['MRR@10']:.4f} | "
#         f"lr {lr_now:.6f} | "
#         f"{epoch_time:.1f}s"
#     )

#     if val_metrics["NDCG@10"] > best_val_ndcg_hybrid:
#         best_val_ndcg_hybrid = val_metrics["NDCG@10"]
#         best_state_hybrid = {k: v.cpu().clone() for k, v in hybrid_model.state_dict().items()}

# total_train_sec_hybrid = time.perf_counter() - train_start_hybrid
# print("\nhybrid fine-tuning finished in", round(total_train_sec_hybrid, 2), "seconds")

In [19]:
total_train_sec_hybrid = sum(row["epoch_sec"] for row in history_hybrid_stage2)

In [ ]:
# Z — evaluate and save hybrid outputs

hybrid_model.load_state_dict(best_state_hybrid)

test_start_hybrid = time.perf_counter()
test_metrics_hybrid = evaluate_topk(hybrid_model, test_loader, device, k=10)
test_sec_hybrid = time.perf_counter() - test_start_hybrid

print("best validation NDCG@10:", round(best_val_ndcg_hybrid, 6))
print("hybrid benchmark test metrics:")
for k, v in test_metrics_hybrid.items():
    print(k, ":", round(v, 6))
print("hybrid benchmark test eval seconds:", round(test_sec_hybrid, 2))

history_hybrid_df = pd.concat([
    pd.DataFrame(history_hybrid_stage1),
    pd.DataFrame(history_hybrid_stage2),
], ignore_index=True)

torch.save(best_state_hybrid, MODEL_DIR / "benchmark_sasrec_hybrid_gated.pt")
history_hybrid_df.to_csv(RESULT_DIR / "benchmark_sasrec_hybrid_gated_history.csv", index=False)

pd.DataFrame([{
    "model": "benchmark_sasrec_hybrid_gated",
    "num_users": num_users,
    "num_items": num_items,
    "max_len": MAX_LEN,
    "d_model": D_MODEL,
    "n_heads": N_HEADS,
    "n_layers": N_LAYERS,
    "dropout": DROPOUT,
    "batch_size": BATCH_SIZE,
    "accum_steps": ACCUM_STEPS,
    "effective_batch_size": BATCH_SIZE * ACCUM_STEPS,
    "label_smoothing": LABEL_SMOOTHING,
    "lr_stage1": LR_STAGE1,
    "lr_stage2_backbone": BACKBONE_LR_STAGE2,
    "lr_stage2_heads": HEAD_LR_STAGE2,
    "weight_decay": WEIGHT_DECAY,
    "epochs_stage1_planned": EPOCHS_STAGE1,
    "epochs_stage2_planned": EPOCHS_STAGE2,
    "train_total_sec_stage1": total_train_sec_stage1,
    "train_total_sec_stage2": total_train_sec_hybrid,
    "train_total_sec_all": total_train_sec_stage1 + total_train_sec_hybrid,
    "test_eval_sec": test_sec_hybrid,
    **test_metrics_hybrid,
}]).to_csv(RESULT_DIR / "benchmark_sasrec_hybrid_gated_test_metrics.csv", index=False)

print("saved model:", MODEL_DIR / "benchmark_sasrec_hybrid_gated.pt")
print("saved history:", RESULT_DIR / "benchmark_sasrec_hybrid_gated_history.csv")
print("saved test metrics:", RESULT_DIR / "benchmark_sasrec_hybrid_gated_test_metrics.csv")

plot_history_curves(pd.DataFrame(history_hybrid_stage1), "hybrid stage1")
plot_history_curves(pd.DataFrame(history_hybrid_stage2), "hybrid stage2")


best validation NDCG@10: 0.027647
hybrid benchmark test metrics:
HR@10 : 0.044291
NDCG@10 : 0.024068
MRR@10 : 0.017944
Recall@10 : 0.044291
hybrid benchmark test eval seconds: 41.59
saved model: ../models/benchmark_sasrec_hybrid_gated.pt
saved history: ../reports/results/benchmark_sasrec_hybrid_gated_history.csv
saved test metrics: ../reports/results/benchmark_sasrec_hybrid_gated_test_metrics.csv


## C. comparison note

Use the strict benchmark as the main table. If a second table uses sampled negatives or a filtered easier setup like `5*ML.20M`, label it as a different protocol and do not merge the numbers into one claim.

That keeps the presentation honest and still lets the results look organized.